In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error


print("--- Step 1: Loading Data ---")
races = pd.read_csv('races.csv')
results = pd.read_csv('results.csv')
lap_times = pd.read_csv('lap_times.csv')
pit_stops = pd.read_csv('pit_stops.csv')


TARGET_RACE_ID = 1014 


finished_results = results[(results['raceId'] == TARGET_RACE_ID) & (results['statusId'] == 1)]


drivers_to_keep = finished_results['driverId'].head(6).tolist()
print(f"Selected Driver IDs who finished: {drivers_to_keep}")


race_laps = lap_times[(lap_times['raceId'] == TARGET_RACE_ID) & (lap_times['driverId'].isin(drivers_to_keep))].copy()
race_pits = pit_stops[(pit_stops['raceId'] == TARGET_RACE_ID) & (pit_stops['driverId'].isin(drivers_to_keep))].copy()

race_laps['lap_time_secs'] = race_laps['milliseconds'] / 1000.0


print("\n--- Step 2: Cleaning Data ---")
initial_lap_count = len(race_laps)


race_laps_cleaned = race_laps[race_laps['lap'] != 1].copy()

pit_in_laps = set(race_pits['lap'])
pit_out_laps = {lap + 1 for lap in pit_in_laps}
laps_to_remove_pit = pit_in_laps.union(pit_out_laps)


race_laps_cleaned = race_laps_cleaned[~race_laps_cleaned['lap'].isin(laps_to_remove_pit)]

final_lap_count = len(race_laps_cleaned)
removed_laps = initial_lap_count - final_lap_count
print(f"Initial lap count: {initial_lap_count}")
print(f"Cleaned lap count: {final_lap_count}")
print(f"Total anomalous laps removed: {removed_laps}")


print("\n--- Step 3: Feature Engineering ---")

processed_driver_data = []

for driver in drivers_to_keep:
    driver_df = race_laps_cleaned[race_laps_cleaned['driverId'] == driver].sort_values('lap').copy()
    driver_pits = race_pits[race_pits['driverId'] == driver]['lap'].tolist()
    
    stints = []
    tire_ages = []
    
    current_stint = 1
    current_tire_age = 1
    
    for idx, row in driver_df.iterrows():
        current_lap = row['lap']
        
        
        passed_pits = sum(1 for pit_lap in driver_pits if current_lap > pit_lap)
        current_stint = 1 + passed_pits
        
        
        if passed_pits == 0:
            current_tire_age = current_lap  
        else:
            last_pit_lap = [pit_lap for pit_lap in driver_pits if current_lap > pit_lap][-1]
            current_tire_age = current_lap - last_pit_lap
            
        stints.append(current_stint)
        tire_ages.append(current_tire_age)
        
    driver_df['stint_number'] = stints
    driver_df['tire_age'] = tire_ages
    processed_driver_data.append(driver_df)


final_df = pd.concat(processed_driver_data).reset_index(drop=True)

print("\n--- Step 4: Stint-Based Train/Test Split ---")


train_data = final_df[final_df['stint_number'] == 1]

test_data = final_df[final_df['stint_number'] == 2]

print(f"Training samples (Stint 1): {len(train_data)}")
print(f"Testing samples (Stint 2): {len(test_data)}")


X_train_baseline = train_data[['lap']]
X_test_baseline = test_data[['lap']]

X_train_degrad = train_data[['lap', 'tire_age']]
X_test_degrad = test_data[['lap', 'tire_age']]

y_train = train_data['lap_time_secs']
y_test = test_data['lap_time_secs']


print("\n--- Step 5: Training & Evaluating Models ---")


baseline_model = LinearRegression()
baseline_model.fit(X_train_baseline, y_train)
y_pred_base = baseline_model.predict(X_test_baseline)


degrad_model = LinearRegression()
degrad_model.fit(X_train_degrad, y_train)
y_pred_degrad = degrad_model.predict(X_test_degrad)

n
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_base = mean_absolute_error(y_test, y_pred_base)

rmse_degrad = np.sqrt(mean_squared_error(y_test, y_pred_degrad))
mae_degrad = mean_absolute_error(y_test, y_pred_degrad)


print("\n--- MODEL PERFORMANCE RESULTS (STINT 2) ---")
print(f"Baseline Model        -> RMSE: {rmse_base:.3f} s | MAE: {mae_base:.3f} s")
print(f"Tire Degradation Model -> RMSE: {rmse_degrad:.3f} s | MAE: {mae_degrad:.3f} s")


print("\n--- Step 6: Generating Degradation Plot ---")


sample_driver_id = drivers_to_keep[0]
driver_test_mask = test_data['driverId'] == sample_driver_id

driver_test_data = test_data[driver_test_mask].sort_values('lap')
driver_laps = driver_test_data['lap']


driver_y_actual = driver_test_data['lap_time_secs']
driver_y_base = y_pred_base[driver_test_mask]
driver_y_degrad = y_pred_degrad[driver_test_mask]


plt.figure(figsize=(10, 6))
plt.plot(driver_laps, driver_y_actual, 'ko-', label='Actual Lap Times', linewidth=1.5)
plt.plot(driver_laps, driver_y_base, 'r--', label='Baseline Model Prediction (Lap Only)', linewidth=2)
plt.plot(driver_laps, driver_y_degrad, 'g-', label='Tire Degradation Model Prediction (Lap + Tire Age)', linewidth=2)

plt.title(f'Formula 1 Performance Tracking: Stint 2 Degradation Curve (Driver ID: {sample_driver_id})')
plt.xlabel('Race Lap Number')
plt.ylabel('Lap Time (Seconds)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.show()